In [ ]:
from google.cloud import bigquery
client = bigquery.Client(project="expanded-flame-491820-j2")

In [13]:
for dataset in client.list_datasets():
    print(f"Dataset: {dataset.dataset_id}")
    for table in client.list_tables(dataset.dataset_id):
        print(f"  Table: {table.table_id}")

Dataset: automobile
  Table: automobile
  Table: automobile_summary
  Table: cairo_weather


In [15]:
from google.cloud import bigquery

client = bigquery.Client(project="expanded-flame-491820-j2")

query = """
    SELECT COUNT(*) as total_rows
    FROM `expanded-flame-491820-j2.automobile.automobile`
"""

df = client.query(query).to_dataframe()
print(df)

C:\Users\PC\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


   total_rows
0         205


In [17]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project="expanded-flame-491820-j2")
PROJECT = "expanded-flame-491820-j2"

In [19]:
# EXTRACT
print("Extracting...")
query = f"""
    SELECT *
    FROM `{PROJECT}.automobile.automobile`
    LIMIT 10
"""
df = client.query(query).to_dataframe()

Extracting...


C:\Users\PC\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [21]:
print("Transforming...")
print(df.columns.tolist()) 

Transforming...
['symboling', 'normalized-losses', 'make', 'fuel-type', 'aspiration', 'num-of-doors', 'body-style', 'drive-wheels', 'engine-location', 'wheel-base', 'length', 'width', 'height', 'curb-weight', 'engine-type', 'num-of-cylinders', 'engine-size', 'fuel-system', 'bore', 'stroke', 'compression-ratio', 'horsepower', 'peak-rpm', 'city-mpg', 'highway-mpg', 'price']


In [23]:
print(df.head())

   symboling normalized-losses         make fuel-type aspiration num-of-doors  \
0          3                 ?  alfa-romero       gas        std          two   
1          3                 ?  alfa-romero       gas        std          two   
2          1                 ?  alfa-romero       gas        std          two   
3          2               164         audi       gas        std         four   
4          2               164         audi       gas        std         four   

    body-style drive-wheels engine-location  wheel-base  ...  engine-size  \
0  convertible          rwd           front        88.6  ...          130   
1  convertible          rwd           front        88.6  ...          130   
2    hatchback          rwd           front        94.5  ...          152   
3        sedan          fwd           front        99.8  ...          109   
4        sedan          4wd           front        99.4  ...          136   

   fuel-system  bore  stroke compression-ratio hor

In [25]:
from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project="expanded-flame-491820-j2")
PROJECT = "expanded-flame-491820-j2"

# EXTRACT
print("Extracting...")
query = f"""
    SELECT *
    FROM `{PROJECT}.automobile.automobile`
"""
df = client.query(query).to_dataframe()
print(f"Extracted {len(df)} rows")

# TRANSFORM
print("Transforming...")

df = df.replace('?', None)

df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['horsepower'] = pd.to_numeric(df['horsepower'], errors='coerce')
df['city-mpg'] = pd.to_numeric(df['city-mpg'], errors='coerce')

df['price_tier'] = df['price'].apply(
    lambda x: 'Budget' if x < 10000 else 'Mid-Range' if x < 20000 else 'Premium' if pd.notna(x) else 'Unknown'
)

df['efficiency_rating'] = df['city-mpg'].apply(
    lambda x: 'High' if x > 30 else 'Medium' if x > 20 else 'Low' if pd.notna(x) else 'Unknown'
)

print(df[['make', 'price', 'price_tier', 'city-mpg', 'efficiency_rating']].head(10))

# LOAD
print("Loading...")
table_id = f"{PROJECT}.automobile.automobile_summary"

job_config = bigquery.LoadJobConfig(
    write_disposition="WRITE_TRUNCATE",
    autodetect=True
)

job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
job.result()

print(f"Done! Loaded {len(df)} rows into {table_id}")

Extracting...


C:\Users\PC\anaconda3\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


Extracted 205 rows
Transforming...
          make    price price_tier  city-mpg efficiency_rating
0  alfa-romero  13495.0  Mid-Range        21            Medium
1  alfa-romero  16500.0  Mid-Range        21            Medium
2  alfa-romero  16500.0  Mid-Range        19               Low
3         audi  13950.0  Mid-Range        24            Medium
4         audi  17450.0  Mid-Range        18               Low
5         audi  15250.0  Mid-Range        19               Low
6         audi  17710.0  Mid-Range        19               Low
7         audi  18920.0  Mid-Range        19               Low
8         audi  23875.0    Premium        17               Low
9         audi      NaN    Unknown        16               Low
Loading...
Done! Loaded 205 rows into expanded-flame-491820-j2.automobile.automobile_summary
